<a href="https://colab.research.google.com/github/glitterfromcintra-cmd/labs/blob/main/____07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 7. Обработка данных из файла. Использование коллекций

ФИО студента: Шеломкова Полина Александровна

Группа студента: ИНБ-б-о-26-2

Вариант: 30

## Задание

В файле содержатся данные о товарах (ID, наименование, категория
(4 шт.), количество, цена, признак просрочки, признак наличия
скидки).
1. Максимальная цена товара для просроченных и непросроченных
товаров.
2. Суммарный объем в валюте товаров без категории с учетом скидки.
3. Самый дорогой товар с учетом скидки в каждой категории.
4. Количество наименований товаров без скидки


## Анализ задания, получение данных

Файл p7_data_30.csv расположен репозитории. Выполним копирование файла в локальное хранилище Google Colab, используя следующую команду:

In [1]:
!wget https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract7/p7_data_30.csv


--2026-11-12 07:02:08--  https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract7/p7_data_30.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47662 (47K) [text/plain]
Saving to: ‘p7_data_30.csv’

p7_data_30.csv      100%[===================>]  46.54K  --.-KB/s    in 0.008s  

2026-11-12 07:02:08 (5.97 MB/s) - ‘p7_data_30.csv’ saved [47662/47662]



In [2]:
F=open('p7_data_30.csv')
n_to_print=10
for line in F:
  n_to_print-=1
  if n_to_print>0:
    print(line)

ID,Name,Category,Count,Price,OutOfDate,WithDiscount

1,"Fish - Atlantic Salmon, Cold",D,16,$60.49,true,true

2,Sauce - Cranberry,B,26,$19.52,true,true

3,Pan Grease,D,22,$20.87,,true

4,Coffee Guatemala Dark,A,10,$31.81,,true

5,Wine - Carmenere Casillero Del,,22,$75.15,false,

6,"Olives - Black, Pitted",,5,$95.04,,

7,Tea Leaves - Oolong,B,17,$37.78,,

8,Wine - Prem Select Charddonany,A,7,$26.36,false,false



In [3]:
F=open('p7_data_30.csv')
header = F.readline().strip().split(',')
print(header)
print('-'*20)
n_to_print=10
for line in F:
  data = line.strip().split(',')
  n_to_print-=1
  if n_to_print>0:
    print(data)

['ID', 'Name', 'Category', 'Count', 'Price', 'OutOfDate', 'WithDiscount']
--------------------
['1', '"Fish - Atlantic Salmon', ' Cold"', 'D', '16', '$60.49', 'true', 'true']
['2', 'Sauce - Cranberry', 'B', '26', '$19.52', 'true', 'true']
['3', 'Pan Grease', 'D', '22', '$20.87', '', 'true']
['4', 'Coffee Guatemala Dark', 'A', '10', '$31.81', '', 'true']
['5', 'Wine - Carmenere Casillero Del', '', '22', '$75.15', 'false', '']
['6', '"Olives - Black', ' Pitted"', '', '5', '$95.04', '', '']
['7', 'Tea Leaves - Oolong', 'B', '17', '$37.78', '', '']
['8', 'Wine - Prem Select Charddonany', 'A', '7', '$26.36', 'false', 'false']
['9', 'Corn Meal', 'C', '15', '$95.68', 'false', 'false']


In [4]:
import csv

all_products = []

with open('p7_data_30.csv', newline='', encoding='utf-8') as F:
    reader = csv.DictReader(F)
    header = reader.fieldnames
    print(header)
    print('-' * 20)

    for row in reader:
        product = {}
        for key in header:
            product[key] = row[key]

        product['ID'] = int(product['ID'])
        product['Count'] = int(product['Count'])
        product['Price'] = float(product['Price'].replace('$', ''))

        product['OutOfDate'] = (product['OutOfDate'].strip().lower() == 'true')
        product['WithDiscount'] = (product['WithDiscount'].strip().lower() == 'true')

        product['Category'] = product['Category'].strip()

        all_products.append(product)

print(all_products[:3])
print('Всего товаров:', len(all_products))

DISCOUNT = 0.10

def price_with_discount(product):
    """Цена товара с учётом скидки (если она есть)."""
    if product['WithDiscount']:
        return product['Price'] * (1 - DISCOUNT)
    return product['Price']

['ID', 'Name', 'Category', 'Count', 'Price', 'OutOfDate', 'WithDiscount']
--------------------
[{'ID': 1, 'Name': 'Fish - Atlantic Salmon, Cold', 'Category': 'D', 'Count': 16, 'Price': 60.49, 'OutOfDate': True, 'WithDiscount': True}, {'ID': 2, 'Name': 'Sauce - Cranberry', 'Category': 'B', 'Count': 26, 'Price': 19.52, 'OutOfDate': True, 'WithDiscount': True}, {'ID': 3, 'Name': 'Pan Grease', 'Category': 'D', 'Count': 22, 'Price': 20.87, 'OutOfDate': False, 'WithDiscount': True}]
Всего товаров: 1000


## Решение задачи 1

Первая задача:
Вывести максимальная цену товара для просроченных и непросроченных товаров.



In [5]:
max_price_out_of_date = 0
max_price_fresh = 0
product_out_of_date = None
product_fresh = None

for product in all_products:
    if product['OutOfDate']:
        if product['Price'] > max_price_out_of_date:
            max_price_out_of_date = product['Price']
            product_out_of_date = product
    else:
        if product['Price'] > max_price_fresh:
            max_price_fresh = product['Price']
            product_fresh = product

print('Максимальная цена просроченного товара: $%.2f' % max_price_out_of_date)
print('  товар:', product_out_of_date['Name'])
print('Максимальная цена непросроченного товара: $%.2f' % max_price_fresh)
print('  товар:', product_fresh['Name'])

Максимальная цена просроченного товара: $99.76
  товар: Country Roll
Максимальная цена непросроченного товара: $99.86
  товар: Pork Loin Bine - In Frenched


## Решение задачи 2

Вторая задача: Суммарный объем в валюте товаров без категории с учетом скидки.

In [6]:
total_no_category = 0

for product in all_products:
    if product['Category'] == '':          # товар без категории
        total_no_category += price_with_discount(product) * product['Count']

print('Суммарный объём в валюте товаров без категории (с учётом скидки): $%.2f'
      % total_no_category)

Суммарный объём в валюте товаров без категории (с учётом скидки): $252787.97


## Решение задачи 3

Третья задача: Самый дорогой товар с учетом скидки в каждой категории.

In [8]:
best_by_category = {}

for product in all_products:
    cat = product['Category']
    if cat == '':
        continue

    current_price = price_with_discount(product)

    if cat not in best_by_category:
        best_by_category[cat] = product
    else:
        best_price = price_with_discount(best_by_category[cat])
        if current_price > best_price:
            best_by_category[cat] = product

print('Самый дорогой товар с учётом скидки в каждой категории:')
for cat, product in sorted(best_by_category.items()):
    print('  Категория %s: %s — $%.2f (со скидкой $%.2f)'
          % (cat, product['Name'], product['Price'], price_with_discount(product)))

Самый дорогой товар с учётом скидки в каждой категории:
  Категория A: Sea Bass - Fillets — $99.25 (со скидкой $99.25)
  Категория B: Country Roll — $99.76 (со скидкой $99.76)
  Категория C: Pork Loin Bine - In Frenched — $99.86 (со скидкой $99.86)
  Категория D: Wine - Stoneliegh Sauvignon — $98.42 (со скидкой $98.42)


## Решение задачи 4

Четвертая задача: Количество наименований товаров без скидки

In [ ]:
count_no_discount = 0

for product in all_products:
    if not product['WithDiscount']:
        count_no_discount += 1

print('Количество наименований товаров без скидки:', count_no_discount)

Количество наименований товаров без скидки: 626


# Контрольные вопросы

1. Напишите код для печати суммы элементов списка p:
p = [10, 100, 34, 56, 45, 45]


In [18]:
p = [10,100,34,56,45,45]
print(sum(p))

290


2. Напишите код для печати количества элементов списка p:
p = [10, 100, 34, 56, 45, 45]


In [20]:
p = [10,100,34,56,45,45]
print(len(p))

6


3. Напишите код для печати суммы элементов кратных 7:
p = [10, 100, 35, 34, 56, 45, 45, 21]


In [21]:
p = [10,100,35,34,56,45,45,21]
print(sum(k for k in p if k%7==0))

112


4. Напишите код для поиска ключей словаря d, для которых значения
превышают 45:
p = {4:10, 12:100, 35:3, 34:56, 56:11, 45:’abc’}


In [23]:
p = {4: 10, 12: 100, 35: 3, 34: 56, 56: 11, 45: 'abc'}
keys = [k for k, v in p.items() if isinstance(v, (int, float)) and v > 45]
print(keys)

[12, 34]


5. Напишите код, который выводит список, составленный из квадратов чисел
от 1 до 100. Сам список также необходимо сгенерировать.

In [24]:
kv = [x**2 for x in range(1, 101)]
print(kv)


[1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400, 441, 484, 529, 576, 625, 676, 729, 784, 841, 900, 961, 1024, 1089, 1156, 1225, 1296, 1369, 1444, 1521, 1600, 1681, 1764, 1849, 1936, 2025, 2116, 2209, 2304, 2401, 2500, 2601, 2704, 2809, 2916, 3025, 3136, 3249, 3364, 3481, 3600, 3721, 3844, 3969, 4096, 4225, 4356, 4489, 4624, 4761, 4900, 5041, 5184, 5329, 5476, 5625, 5776, 5929, 6084, 6241, 6400, 6561, 6724, 6889, 7056, 7225, 7396, 7569, 7744, 7921, 8100, 8281, 8464, 8649, 8836, 9025, 9216, 9409, 9604, 9801, 10000]



6. Опишите структуру файла csv. Поясните последовательность действий при
чтении файла csv.

CSV - текстовый файл для представления табличных данных. Он состоит из строк, столбцов и заголовков.
Чтение csv происходит следующим образом:
1) Импорт модуля. В начале программы подключается встроенная библиотека командой import csv.
2) Открытие файла. Файл открывается на чтение с помощью менеджера контекста with open(). Обязательно указываются режим чтения 'r', правильная кодировка (encoding='utf-8') и параметр newline='', который предотвращает проблемы с пустыми строками при обработке переносов.
3) Создание объекта для чтения. Открытый файловый объект передается в функцию csv.reader() (возвращает данные в виде списков) или csv.DictReader() (возвращает данные в виде словарей, где ключами выступают заголовки). На этом этапе при необходимости указывается нестандартный разделитель, например delimiter=';'.
4) Построчное чтение данных. С помощью цикла for программа поочередно считывает строки из созданного объекта. Если используется csv.reader и в файле есть заголовок, его можно пропустить перед циклом с помощью функции next().
5) Обработка данных и закрытие файла. Внутри цикла полученные значения преобразуются в нужные типы (например, строки в числа) и сохраняются или выводятся. После выхода из блока with файл закрывается автоматически.

7. Предположим, что 'старт': 1472 является элементом словаря. Что является ключом? И что является значением?

Ключом будет 'Старт', а значением 1472

8. Предположим, что создан словарь с именем employee. Что делает
приведенная ниже инструкция?
```
employee['id'] = 54321
```
9. Что покажет приведенный ниже фрагмент кода?
```
stuff = {1:'ааа', 2:'ббб', 3: 'ввв'}
print(stuff[3])
```

In [17]:
stuff = {1:'ааа', 2:'ббб', 3: 'ввв'}
print(stuff[3])

ввв


10. Как определить, существует ли пара "ключ : значение" в словаре?

В Python проверить существование конкретной пары «ключ : значение» в словаре можно с помощью оператора in вместе с методом .items()

11. Предположим, что существует словарь inventory. Что делает
приведенная ниже инструкция?
```
del inventory[654]
```
Эта инструкция удаляет из словаря inventory элемент (пару «ключ — значение») с ключом 654

12. Что покажет приведенный ниже фрагмент кода?
```
stuff = {1:'ааа', 2:'ббб’, 3:'ввв'}
print(len(stuff))
```


In [16]:
stuff = {1:'ааа', 2:'ббб', 3:'ввв'}
print(len(stuff))

3


13. Что покажет приведенный ниже фрагмент кода?
```
stuff = {1:'ааа', 2:'ббб', 3:'ввв'}
for k in stuff:
print(к)
```

In [13]:
stuff = {1:'ааа', 2:'ббб', 3:'ввв'}
for k in stuff:
  print(k)

1
2
3


14. В чем разница между словарными методами pop() и popitem() применительно к словарям?

* Метод pop(key[, default]): Удаляет элемент с указанным ключом key и возвращает только его значение. Если ключ не найден, метод вызовет ошибку KeyError, если не передано запасное значение default.
* Метод popitem(): Не принимает никаких аргументов и удаляет последний добавленный в словарь элемент. Он возвращает удаленную пару в виде кортежа (ключ, значение), а при вызове на пустом словаре вызывает ошибку KeyError.

15. Что возвращает метод items()?

Метод items() возвращает специальный объект представления (dict_items), который содержит все пары «ключ — значение» словаря в виде кортежей. Этот объект динамически связан со словарем и автоматически обновляется при любых изменениях в нем.

16. Что возвращают методы keys() и values() словаря?

Оба метода возвращают динамические объекты представления (view objects), которые позволяют просматривать содержимое словаря без создания его полной копии в памяти. При изменении исходного словаря эти объекты сразу отражают новые данные.
* Метод keys(): Возвращает объект типа dict_keys, содержащий список всех ключей
словаря.
* Метод values(): Возвращает объект типа dict_values, содержащий список всех значений словаря.
